In [2]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [3]:
import os
from pathlib import Path

human_in_the_loop_root = Path().absolute().parent

os.chdir(human_in_the_loop_root)

Path().absolute()

PosixPath('/home/agaros/mywork/NWDAF-Anomaly-Detection/human_in_the_loop')

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
from artifact_experiment.tracking import DataSplit, FilesystemTrackingClient
from artifact_torch.binary_classification import BinaryClassSpec
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split

# from demos.binary_classification.config.constants import (
#     EXPERIMENT_ID,
#     LABEL_FEATURE,
#     LS_CLASS_NAMES,
#     LS_FEATURES,
#     POSITIVE_CLASS_NAME,
#     TRAINING_DATASET_PATH,
#     VAL_DATA_PROPORTION,
# )

# from demos.binary_classification.data.utils import DemoDataUtils
from hitl.experiment.experiment import DemoBinaryClassificationExperiment
from hitl.models.ae_artifact_ml import FilterModel
from hitl.training.trainer import Trainer
from hitl.store.sqlite import SQLite
from hitl.store.repository import Repository
from hitl.io.serialization import tensor_from_list, encode_npy, decode_npy, validate_tensor
from hitl.experiment.data.dataset import FilterModelClassifierDataset


In [5]:
EXPERIMENT_ID = "demo-human-in-the-loop"
VAL_DATA_PROPORTION = 0.2


POSITIVE_CLASS = "anomaly"
NEGATIVE_CLASS = "normal"
CLASSNAMES = [NEGATIVE_CLASS, POSITIVE_CLASS]

LABEL_FEATURE = "true_label"

In [6]:
sns.set_theme(style="whitegrid", palette="colorblind")

In [7]:
# DB setup

db_path = human_in_the_loop_root / "db" / "hitl.db"

sqlite_store = SQLite(path=str(db_path))

repo = Repository(sqlite_store)

In [8]:
def load_training_dataset(
    repo, schema_id: str, mode: str, only_false_positives: bool = True
) -> tuple[np.ndarray, list[str], tuple[int, ...], list[str], dict[str, tuple[np.ndarray, str]]]:
    """Load all vectors for a schema from database.

    Args:
        schema_id: Feature schema to load
        mode: "dense" or "conv1d" - determines shape transformation

    Returns:
        (X, ids, storage_sample_shape) tuple where:
        - X: Data array
            - Dense mode: X has shape (N, D)
            - Conv1d mode: X has shape (N, F, T) - transposed from storage format
        - ids: List of anomaly_ids
        - storage_sample_shape: Original sample shape in storage format
            - Dense: (D,)
            - Conv1d: (T, F)

    Shape Convention:
        Storage format (per sample):
            - Dense: (D,) - feature vector
            - Conv1d: (T, F) - time-series with T timesteps, F features
        
        Training format (batch):
            - Dense: (N, D) - no transformation needed
            - Conv1d: (N, F, T) - transposed from (N, T, F) for PyTorch Conv1d
                PyTorch Conv1d expects channels-first format (B, C, L)

    Raises:
        ValueError: if no vectors found for schema
    """
    # print("Loading dataset", schema_id=schema_id, mode=mode)

    vectors = []
    ids = []
    labels = []
    all = {}
    
    # Iterate over all vectors for this schema
    for anomaly_id, blob in repo.iter_vectors(schema_id):
        # If requested, filter to only include samples labeled as false positives
        fb = repo.latest_feedback(anomaly_id)
        if only_false_positives:
            if not fb or fb["label"] != "FP":
                continue
        else:
            if not fb or fb["label"] not in ["TP", "FP"]:
                continue  # Skip samples with no feedback

        # Decode the .npy blob (storage format expected: (T, F) for conv1d)
        arr = decode_npy(blob)
        # Basic validation to catch unexpected shapes early
        if mode == "conv1d" and getattr(arr, "ndim", None) != 2:
            raise ValueError(
                f"Expected 2D array for conv1d storage format (T,F), got ndim={getattr(arr, 'ndim', None)}"
            )
        if mode == "dense" and getattr(arr, "ndim", None) != 1:
            raise ValueError(
                f"Expected 1D array for dense storage format (D,), got ndim={getattr(arr, 'ndim', None)}"
            )

        vectors.append(arr)
        ids.append(anomaly_id)
        labels.append(fb["label"])
        all[str(anomaly_id)] = (arr, fb["label"])
        # id is string with structure train-INTEGER
        # convert to integer
        # int_id = int(anomaly_id.split("-")[1])
        # all[int(int_id)] = (arr, fb["label"])
    if not vectors:
        raise ValueError(f"No vectors found for schema {schema_id}")

    # Stack into single array
    X = np.stack(vectors, axis=0)

    # Store original sample shape (storage format) for model building
    storage_sample_shape = X.shape[1:]  # Shape per sample in storage format

    # For Conv1d mode, transpose from storage format (N, T, F) to training format (N, F, T)
    # PyTorch Conv1d expects (Batch, Channels, Length) = (N, F, T)
    if mode == "conv1d" and X.ndim == 3:
        X = np.transpose(X, (0, 2, 1))  # (N, T, F) -> (N, F, T)

    return X, ids, storage_sample_shape, labels, all

In [9]:
ds = load_training_dataset(repo, schema_id="67b4d11c626dc4c211ff758c0667e294ac3fb145", mode="conv1d", only_false_positives=False)

In [10]:
ds[2]

(12, 8)

In [11]:
all_dict = ds[4]

# validation_set should

In [12]:
from hitl.io.serialization import decode_npy
import numpy as np
schema_id = "67b4d11c626dc4c211ff758c0667e294ac3fb145"
print('Inspecting first decoded sample for schema', schema_id)
for anomaly_id, blob in repo.iter_vectors(schema_id):
    try:
        arr = decode_npy(blob)
    except Exception as e:
        print('Error decoding blob for', anomaly_id, '->', e)
        break
    print('anomaly_id:', anomaly_id)
    print('decoded arr shape (raw):', getattr(arr, 'shape', None), 'ndim=', getattr(arr, 'ndim', None))
    if getattr(arr, 'ndim', None) == 2:
        print('interpreted as (T, F) storage shape =', arr.shape)
    break

# Build a small stack of up to 10 samples to inspect batch shapes
blobs = [blob for _, blob in list(repo.iter_vectors(schema_id))[:10]]
if blobs:
    try:
        X = np.stack([decode_npy(b) for b in blobs], axis=0)
        print('stacked X shape (before transpose):', X.shape)
        if X.ndim == 3:
            Xt = np.transpose(X, (0,2,1))
            print('stacked X shape (after transpose to (N,F,T)):', Xt.shape)
    except Exception as e:
        print('Error stacking/inspecting blobs ->', e)

Inspecting first decoded sample for schema 67b4d11c626dc4c211ff758c0667e294ac3fb145
anomaly_id: train-00000000
decoded arr shape (raw): (12, 8) ndim= 2
interpreted as (T, F) storage shape = (12, 8)
stacked X shape (before transpose): (10, 12, 8)
stacked X shape (after transpose to (N,F,T)): (10, 8, 12)


In [13]:
from __future__ import annotations
from typing import Dict, Tuple
import numpy as np

Array = np.ndarray
Sample = Tuple[Array, str]
Dataset = Dict[str, Sample]


def split_fp_tp(
    all_data: Dataset,
    fp_val_percentage: float,
    rng: np.random.Generator | None = None,
) -> tuple[Dataset, Dataset]:
    """
    Split a dict[id -> (features, label)] into:
      1) FP-only train set
      2) validation set with FP subset + all TPs

    Args:
        all_data: mapping id -> (features, label), label in {"FP", "TP"}.
        fp_val_percentage: proportion of FP samples to send to validation
                           (between 0.0 and 1.0).
        rng: optional numpy random generator for reproducibility.

    Returns:
        train_fp_only, val_fp_and_tp
    """
    if not (0.0 <= fp_val_percentage <= 1.0):
        raise ValueError("fp_val_percentage must be in [0.0, 1.0].")

    if rng is None:
        rng = np.random.default_rng()

    # Collect IDs by label
    fp_ids = [sid for sid, (_, label) in all_data.items() if label == "FP"]
    tp_ids = [sid for sid, (_, label) in all_data.items() if label == "TP"]

    # Decide how many FPs to send to validation
    n_fp = len(fp_ids)
    n_fp_val = int(np.floor(n_fp * fp_val_percentage))

    # Shuffle and split FP IDs
    fp_ids_array = np.array(fp_ids)
    rng.shuffle(fp_ids_array)

    fp_val_ids = set(fp_ids_array[:n_fp_val])
    fp_train_ids = set(fp_ids_array[n_fp_val:])

    # Build train (FP-only)
    train_fp_only: Dataset = {
        sid: all_data[sid]
        for sid in fp_train_ids
    }

    # Build validation: selected FPs + all TPs
    val_ids = fp_val_ids.union(tp_ids)
    val_fp_and_tp: Dataset = {
        sid: all_data[sid]
        for sid in val_ids
    }

    return train_fp_only, val_fp_and_tp

In [14]:
train_set, validation_set = split_fp_tp(all_dict, fp_val_percentage=0.2)

In [15]:
# train_set

In [16]:
class_spec = BinaryClassSpec(
    class_names=CLASSNAMES, positive_class=POSITIVE_CLASS, label_name=LABEL_FEATURE
)

class_spec

BinaryClassSpec(label_name='true_label', classes=['normal', 'anomaly'], positive_class='anomaly', negative_class='normal')

In [17]:
train_set, validation_set = split_fp_tp(all_dict, fp_val_percentage=0.2)
arrays = {
    DataSplit.TRAIN: np.stack([entry[0] for entry in train_set.values()]),
    DataSplit.VALIDATION: np.stack([entry[0] for entry in validation_set.values()])
}

# arrays


In [18]:
# arrays[DataSplit.TRAIN]

In [19]:
datasets = {
    data_split: FilterModelClassifierDataset(input_array=array) for data_split, array in arrays.items()
}
datasets[DataSplit.TRAIN][0]

{'initial_tensor': tensor([[ 2.2151,  3.9430,  2.4007,  3.3601,  3.0383,  1.4937,  1.6186, -0.8804],
         [-1.1902,  0.0936, -1.1397, -0.9347, -5.3812,  0.6399, -0.7207, -1.2737],
         [ 1.9842,  3.1731,  1.6694, -0.4789, -2.9322,  0.9994,  1.5487,  0.4931],
         [ 1.0859,  0.5369,  1.3628,  1.5268,  0.2854,  1.2990,  1.3298,  0.2565],
         [ 2.1865,  1.3534,  2.1661,  2.3020,  0.3927,  1.7184,  1.6972,  0.6145],
         [ 2.0977, -0.9562,  1.8718,  0.7169, -2.1457,  2.1228,  1.4214,  2.4781],
         [ 1.9764,  3.0332,  1.6510, -0.4735, -2.9322,  1.2690,  1.5637,  0.4439],
         [ 2.2195,  3.2781,  2.1109,  3.4091,  2.8596,  1.2915,  1.6985, -1.2461],
         [-1.1992, -1.2012, -1.2926, -0.9982, -0.4653, -0.4386, -0.3712,  0.0506],
         [ 2.0819,  3.3831,  1.9315,  0.5995,  3.0383,  0.7972,  1.1293, -0.8528],
         [-1.1983, -1.2012, -1.2834, -0.9980, -0.4296, -0.2813, -0.3787,  0.2842],
         [-1.2003, -1.1662, -1.2926, -1.0210,  3.1456, -1.3598, -1.56

In [20]:
from artifact_torch.nn import DataLoader

data_loaders = {
    data_split: DataLoader(dataset=dataset, batch_size=32, shuffle=True, drop_last=True) for data_split, dataset in datasets.items()
}

data_loaders[DataSplit.TRAIN].__iter__().__next__()

{'initial_tensor': tensor([[[-1.1992, -1.2012, -1.2926,  ..., -0.3263, -0.3712,  0.2381],
          [ 0.4616, -0.9387,  0.3850,  ...,  0.6624,  0.6249,  1.4426],
          [-1.2005, -0.5013, -1.2834,  ..., -1.0453, -1.5571,  2.3214],
          ...,
          [-1.1992, -1.2012, -1.2926,  ..., -0.4161, -0.3587, -0.0078],
          [-1.1990, -1.1662, -1.2903,  ..., -0.2589, -0.3737,  0.0629],
          [-1.2010,  0.4086, -1.2995,  ..., -1.1576, -1.5596,  0.8311]],
 
         [[ 2.2983,  6.1127,  3.9622,  ...,  1.7858,  3.1815, -1.2584],
          [ 2.1949,  3.4881,  2.1431,  ...,  1.3814,  1.5737, -1.4059],
          [ 0.3916,  1.1085,  0.2183,  ..., -0.0117,  0.0357, -1.0725],
          ...,
          [-1.1990, -0.5713, -1.2696,  ..., -1.2924, -1.5571, -1.1140],
          [-1.2000, -1.0962, -1.2880,  ..., -1.3598, -1.5596, -1.9344],
          [-1.1992, -1.2012, -1.2926,  ..., -0.4611, -0.3862, -0.0293]],
 
         [[ 1.9892,  3.0682,  1.7959,  ...,  0.9994,  1.5287,  1.3105],
          

In [21]:
from artifact_torch.binary_classification import BinaryClassStore, BinaryClassificationRoutineData

# id dictionaries

def _map_labels(label):
    if label == "FP":
        return NEGATIVE_CLASS
    elif label == "TP":
        return POSITIVE_CLASS
    else:
        raise ValueError(f"Unknown label: {label}")


id_to_class_true_train = {
    sid: _map_labels(label) for sid, (_, label) in train_set.items()
}

id_to_class_true_val = {
    sid: _map_labels(label) for sid, (_, label) in validation_set.items()
}

true_labels = { DataSplit.TRAIN: id_to_class_true_train, DataSplit.VALIDATION: id_to_class_true_val }

true_label_stores = {
    data_split: BinaryClassStore.from_class_names_and_spec(
        class_spec=class_spec, id_to_class=id_to_class
    ) for data_split, id_to_class in true_labels.items()
}

id_to_data_train = {
    sid: features for sid, (features, _) in train_set.items()
}

id_to_data_val = {
    sid: features for sid, (features, _) in validation_set.items()
}

classification_data = { 
    DataSplit.TRAIN: id_to_data_train,
    DataSplit.VALIDATION: id_to_data_val
}   

artifact_routine_data = {
    data_split: BinaryClassificationRoutineData(
        true_class_store=true_label_store,
        classification_data=classification_data[data_split]
    ) for data_split, true_label_store in true_label_stores.items()
}


In [22]:
type(artifact_routine_data)

dict

In [31]:
from hitl.models.ae import build_model, model_summary
# Build model

# conv1d_model = build_model(
#     mode="conv1d",
#     input_shape=(arrays[DataSplit.TRAIN].shape[2], arrays[DataSplit.TRAIN].shape[1]),
#     latent_dim=32, 
# )

conv1d_model = build_model(
    mode="conv1d",
    input_shape=(arrays[DataSplit.TRAIN].shape[2], arrays[DataSplit.TRAIN].shape[1]),
    latent_dim=4, 
    num_filters=[8, 4]
    
)

# print model summary
# print(conv1d_model)
print(model_summary(conv1d_model, input_shape=(arrays[DataSplit.TRAIN].shape[2], arrays[DataSplit.TRAIN].shape[1])))

filter_model = FilterModel(model=conv1d_model)

Building Conv1dAE with seq_len=8, num_features=12
Model: Conv1dAE
Layer (type)                   Output Shape              Param #        
Conv1d                         (1, 8, 4)                 296            
ReLU                           (1, 8, 4)                 0              
Conv1d                         (1, 4, 2)                 100            
ReLU                           (1, 4, 2)                 0              
Linear                         (1, 4)                    36             
Linear                         (1, 8)                    40             
ConvTranspose1d                (1, 8, 4)                 104            
ReLU                           (1, 8, 4)                 0              
ConvTranspose1d                (1, 12, 8)                300            
Total params: 876
Trainable params: 876
Non-trainable params: 0


In [32]:
tracking_client = FilesystemTrackingClient.build(experiment_id=EXPERIMENT_ID)

In [33]:
experiment = DemoBinaryClassificationExperiment.build(
    model=filter_model,
    data_loaders=data_loaders,
    artifact_routine_data=artifact_routine_data,
    artifact_routine_data_spec=class_spec,
    tracking_client=tracking_client,
)


In [34]:
experiment.run()

Training on device: cuda


(Epoch 0):   0%|          | 0/368 [00:00<?, ?it/s]

                                                                          /home/agaros/mywork/NWDAF-Anomaly-Detection/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/agaros/mywork/NWDAF-Anomaly-Detection/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/agaros/mywork/NWDAF-Anomaly-Detection/venv/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/agaros/mywork/NW

KeyboardInterrupt: 

In [ ]:
# # Diagnostics and short smoke-run (small subset)
# from collections import Counter
# import numpy as np

# print('Total samples (all_dict):', len(all_dict))
# print('Label counts (all):', Counter(label for _, label in all_dict.values()))
# print('Train/Val sizes:', len(train_set), len(validation_set))
# print('Train label counts:', Counter(label for _, label in train_set.values()))
# print('Val label counts:', Counter(label for _, label in validation_set.values()))
# print('Arrays shapes (train, val):', arrays[DataSplit.TRAIN].shape, arrays[DataSplit.VALIDATION].shape)

# # Build small subsets for a quick smoke test to avoid long runs
# def _subset_dict(d, n):
#     keys = list(d.keys())[:n]
#     return {k: d[k] for k in keys}

# N_TRAIN = 256
# N_VAL = 256
# train_small = _subset_dict(train_set, N_TRAIN)
# val_small = _subset_dict(validation_set, N_VAL)

# arrays_small = {
#     DataSplit.TRAIN: np.stack([entry[0] for entry in train_small.values()]) if train_small else np.empty((0,) + arrays[DataSplit.TRAIN].shape[1:]),
#     DataSplit.VALIDATION: np.stack([entry[0] for entry in val_small.values()]) if val_small else np.empty((0,) + arrays[DataSplit.VALIDATION].shape[1:])
# }

# from artifact_torch.nn import DataLoader
# datasets_small = {ds: FilterModelClassifierDataset(input_array=arr) for ds, arr in arrays_small.items()}
# data_loaders_small = {ds: DataLoader(dataset=ds_obj, batch_size=32, shuffle=True, drop_last=True) for ds, ds_obj in datasets_small.items()}

# # Use a separate tracking client id to avoid mixing artifacts with main experiment
# tracking_client_small = FilesystemTrackingClient.build(experiment_id=EXPERIMENT_ID + '-smoke')
# experiment_small = DemoBinaryClassificationExperiment.build(
#     model=filter_model,
#     data_loaders=data_loaders_small,
#     artifact_routine_data=artifact_routine_data,
#     artifact_routine_data_spec=class_spec,
#     tracking_client=tracking_client_small,
# )

# print('Running short smoke experiment...')
# experiment_small.run()